
# Analisis Data Mentah — Skrining Sleman 2025

Notebook ini **hanya menganalisis** data mentah `Skrining_Sleman_2025.csv`. Tidak ada
pembersihan, tidak ada modifikasi `df`, dan tidak ada ekspor. Semua perbaikan dilakukan di
notebook berikutnya (`Persiapan_Data_Revised.ipynb`).

Tujuannya menjawab tiga pertanyaan sebelum data dipakai untuk analisis maupun modeling:

1. Apa isi datanya — struktur, variabel, dan populasi yang tercakup?
2. Masalah kualitas apa saja yang ada, dan seberapa besar dampaknya?
3. Apakah ada sifat data yang membatasi kesimpulan yang boleh ditarik nanti?

**Alur:**

| Bagian | Isi |
|---|---|
| 1 | Struktur & kelengkapan data |
| 2 | Masalah pada variabel numerik |
| 3 | Variabel kategorik & deteksi typo |
| 4 | **Gerbang eligibilitas** — arti `Tidak Dapat Pertanyaan` |
| 5 | Struktur kolom `id` (anonimisasi) |
| 6 | Pola tinggi/berat tertukar |
| 7 | Batas plausibilitas per kelompok usia |
| 8 | Ketergantungan label target pada IMT |
| 9 | Korelasi antar variabel |
| 10 | Ringkasan masalah & pemetaan ke tahap pembersihan |

In [ ]:

# ===================== IMPORTS =====================
import os
import sys
from pathlib import Path
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modul bersama + konfigurasi wilayah. Akar proyek dicari dari folder kerja,
# jadi tidak ada satu pun jalur absolut di notebook ini: seluruh folder boleh
# dipindah atau diserahkan ke orang lain tanpa sel ini perlu diubah.
WILAYAH = "Sleman"

AKAR = Path.cwd()
while not (AKAR / "Bersama").exists():
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "Bersama"))

import konfigurasi as cfg      # jalur + cara baca berkas mentah tiap wilayah
import bmi_standards as bs      # standar BMI & batas plausibilitas per usia
import viz_style as vz          # gaya grafik seragam untuk semua notebook

vz.apply_style()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [ ]:

# ===================== LOAD DATA =====================
FILEDATA = cfg.berkas_mentah(WILAYAH)

# Format berkas mentah BERBEDA antar wilayah: Bantul UTF-16 dengan
# pemisah TAB, Sleman UTF-8 dengan pemisah koma. Argumen bacanya
# diambil dari konfigurasi, tidak diketik ulang di sini.
with open(FILEDATA, "rb") as f:
    print("Byte awal    :", f.read(4))
    print("Argumen baca :", cfg.WILAYAH_DATA[WILAYAH]["baca"])

df = cfg.baca_mentah(WILAYAH)

print(f"Shape : {df.shape[0]:,} baris x {df.shape[1]} kolom")
display(df.head())


## 1. Struktur & Kelengkapan Data

In [ ]:

# ===================== INFO & MISSING VALUES =====================
df.info()

missing = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing (%)": (df.isna().mean() * 100).round(2),
})
display(missing)

print(f"Total sel kosong: {int(df.isna().sum().sum())}")


**Tidak ada satu pun nilai kosong.** Ini penting dan sedikit menyesatkan: ketiadaan `NaN`
di sini **bukan** berarti datanya bersih. Seperti terlihat di Bagian 2, nilai yang tidak
masuk akal justru tersimpan sebagai angka valid (misalnya tinggi badan `998`), bukan
sebagai nilai kosong. Jadi pengecekan missing value saja tidak cukup untuk menilai
kualitas data ini.


## 2. Masalah pada Variabel Numerik

In [ ]:

# ===================== SUMMARY NUMERIK =====================
NUM_COLS = ["Usia", "Tinggibadan", "Berat Badan", "Index Masa Tubuh"]
display(df[NUM_COLS].describe().T)


Tiga hal langsung terlihat janggal:

- `Tinggibadan` minimum **negatif** dan maksimum mendekati **998 cm**.
- `Berat Badan` maksimum mendekati **988 kg**.
- `Index Masa Tubuh` punya mean yang jauh lebih besar daripada median, dengan maksimum
  mencapai **jutaan**.

IMT yang meledak adalah *efek berantai*, bukan masalah independen: IMT = berat / tinggi²,
jadi satu tinggi badan yang salah input otomatis merusak IMT-nya. Ini juga sebabnya
rata-rata IMT mentah tidak bisa dipakai untuk deskripsi apa pun sebelum dibersihkan.

In [ ]:

# ===================== SEBERAPA PARAH DAMPAKNYA =====================
imt = df["Index Masa Tubuh"]
print(f"Mean IMT mentah   : {imt.mean():,.1f}")
print(f"Median IMT mentah : {imt.median():,.1f}")
print(f"Mean IMT setelah dibatasi ke rentang wajar 8-70: "
      f"{imt[imt.between(bs.BMI_MIN, bs.BMI_MAX)].mean():.1f}")
print()

# Berapa banyak baris yang benar-benar bermasalah?
for col, (lo, hi) in [("Tinggibadan", bs.GLOBAL_BOUNDS["tinggi"]),
                      ("Berat Badan", bs.GLOBAL_BOUNDS["berat"]),
                      ("Index Masa Tubuh", (bs.BMI_MIN, bs.BMI_MAX))]:
    bad = ~df[col].between(lo, hi)
    print(f"{col:<18}: {bad.sum():>6,} baris di luar [{lo}, {hi}]  ({bad.mean()*100:.3f}%)")

In [ ]:

# ===================== BOXPLOT & HISTOGRAM (RAW) =====================
fig, axes = plt.subplots(2, 4, figsize=vz.FIG_2X4)

for i, col in enumerate(NUM_COLS):
    sns.boxplot(y=df[col], ax=axes[0, i], color="lightcoral")
    axes[0, i].set_title(f"{col}\n(boxplot, skala penuh)")

    # Histogram dibatasi ke persentil 1-99 supaya bentuk distribusinya masih terbaca;
    # kalau pakai skala penuh, semua data menumpuk di satu batang karena outlier ekstrem.
    lo, hi = df[col].quantile([0.01, 0.99])
    sns.histplot(df[col].clip(lo, hi), bins=40, ax=axes[1, i], color="lightcoral")
    axes[1, i].set_title(f"{col}\n(histogram, dipotong p1-p99)")

plt.tight_layout()
plt.show()

In [ ]:

# ===================== CONTOH NILAI EKSTREM =====================
print("10 tinggi badan terbesar:")
display(df.nlargest(10, "Tinggibadan")[["id", "Usia", "Tinggibadan", "Berat Badan", "Index Masa Tubuh"]])

print("10 tinggi badan terkecil:")
display(df.nsmallest(10, "Tinggibadan")[["id", "Usia", "Tinggibadan", "Berat Badan", "Index Masa Tubuh"]])


## 3. Variabel Kategorik & Deteksi Typo

In [ ]:

# ===================== VALUE COUNTS KATEGORIK =====================
TARGETS = [
    "Berisiko Diabetes Melitus",
    "Berisiko Hipertensi",
    "Berisiko Kanker Paru",
    "Berisiko TBC",
]
CAT_COLS = ["Jeniskelamin", "segmen"] + TARGETS

for col in CAT_COLS:
    print(f"\n===== {col} =====")
    print(df[col].value_counts(dropna=False).to_string())


### Distribusi Kelas Awal per Penyakit

Grafik berikut memperlihatkan komposisi **asli** ketiga label pada data mentah, sebelum
pembersihan apa pun dilakukan. Ini titik acuan untuk membandingkan setiap perubahan di
tahap berikutnya.

In [ ]:

# ===================== DISTRIBUSI KELAS AWAL (3 LABEL) =====================
LABEL_ORDER = ["Berisiko", "Tidak Berisiko", "Tidak Dapat Pertanyaan"]

# Typo digabung ke label yang benar HANYA untuk keperluan grafik ini,
# supaya komposisinya tidak terpecah. df aslinya tidak diubah.
def hitung_label(col):
    s = df[col].replace({"Tidah Berisiko": "Tidak Berisiko"})
    return s.value_counts().reindex(LABEL_ORDER).fillna(0).astype(int)


fig, axes = plt.subplots(2, 2, figsize=vz.FIG_2X2)

for ax, col in zip(axes.flat, TARGETS):
    counts = hitung_label(col)
    ax.bar(range(len(LABEL_ORDER)), counts.values,
           color=[vz.PALETTE["RISK"][l] for l in LABEL_ORDER])
    ax.set_xticks(range(len(LABEL_ORDER)))
    ax.set_xticklabels(["Berisiko", "Tidak\nBerisiko", "Tidak Dapat\nPertanyaan"])
    ax.set_title(col.replace("Berisiko ", ""))
    ax.set_ylabel("Jumlah catatan skrining")
    vz.annotate_bars(ax, pct_of=len(df))
    vz.thousands(ax)

fig.suptitle("Distribusi Kelas Awal per Penyakit (data mentah, sebelum pembersihan)",
             y=1.00)
plt.tight_layout()
plt.show()

In [ ]:

# ===================== PERBANDINGAN ANTAR PENYAKIT =====================
komposisi = pd.DataFrame({c.replace("Berisiko ", ""): hitung_label(c) for c in TARGETS}).T
display(komposisi)

persen = (komposisi.div(komposisi.sum(axis=1), axis=0) * 100).round(2)
display(persen)

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

# Kiri: jumlah absolut, berdampingan -- memperlihatkan beda ukuran populasi.
komposisi.plot(kind="bar", ax=axes[0],
               color=[vz.PALETTE["RISK"][l] for l in LABEL_ORDER])
axes[0].set_title("Jumlah Absolut per Label")
axes[0].set_ylabel("Jumlah catatan skrining")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="Label", fontsize=10)
vz.thousands(axes[0])

# Kanan: proporsi 100% -- memperlihatkan komposisi relatif.
persen.plot(kind="bar", stacked=True, ax=axes[1],
            color=[vz.PALETTE["RISK"][l] for l in LABEL_ORDER])
axes[1].set_title("Komposisi Relatif (100%)")
axes[1].set_ylabel("% dari seluruh data")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Label", fontsize=10, loc="center left",
               bbox_to_anchor=(1.01, 0.5))
axes[1].set_ylim(0, 100)

for i, nama in enumerate(persen.index):
    bawah = 0
    for lab in LABEL_ORDER:
        v = persen.loc[nama, lab]
        if v > 6:
            axes[1].text(i, bawah + v / 2, f"{v:.1f}%", ha="center",
                         va="center", fontsize=10, color="white", weight="bold")
        bawah += v

plt.tight_layout()
plt.show()


Dua hal yang langsung terlihat dan menentukan arah seluruh proyek:

1. **Proporsi `Tidak Dapat Pertanyaan` sangat berbeda antar penyakit** — nyaris tidak ada
   pada TBC, tetapi mendominasi pada Kanker Paru. Perbedaan sebesar ini tidak mungkin
   disebabkan kegagalan pencatatan yang acak. Bagian 4 menunjukkan penyebabnya: label itu
   adalah **gerbang eligibilitas usia**, bukan data hilang.

2. **Ketimpangan kelasnya sangat berbeda.** Hipertensi justru didominasi kelas `Berisiko`,
   sedangkan TBC dan Kanker Paru sangat timpang ke arah `Tidak Berisiko`. Perbedaan ini
   yang nanti menentukan pemilihan metrik dan penanganan ketimpangan pada tahap pemodelan.

In [ ]:

# ===================== DETEKSI TYPO OTOMATIS =====================
# Tidak di-hardcode: label apa pun di luar daftar valid dicocokkan secara fuzzy,
# supaya typo baru yang belum diketahui tetap ketahuan.
VALID_LABELS = {"Berisiko", "Tidak Berisiko", "Tidak Dapat Pertanyaan"}

found = []
for col in TARGETS:
    for val in sorted(set(df[col].dropna()) - VALID_LABELS):
        best = max(VALID_LABELS, key=lambda v: SequenceMatcher(None, val, v).ratio())
        score = SequenceMatcher(None, val, best).ratio()
        found.append({"Kolom": col, "Label tidak valid": val,
                      "Kemungkinan benar": best, "Similarity": round(score, 3),
                      "Jumlah baris": int((df[col] == val).sum())})

display(pd.DataFrame(found) if found else "Tidak ada label di luar daftar valid.")


Hanya satu typo yang muncul, tetapi jumlah barisnya besar sehingga tidak bisa diabaikan.
Perlu dicatat bahwa perbaikan typo harus diterapkan ke **semua** kolom target, bukan hanya
kolom tempat typo itu pertama kali terlihat.


## 4. Gerbang Eligibilitas — Arti `Tidak Dapat Pertanyaan`

Label `Tidak Dapat Pertanyaan` muncul dalam jumlah sangat besar (sampai 59% pada Kanker
Paru). Sebelum memutuskan cara memperlakukannyanya, perlu dipastikan dulu: apakah ini
data hilang, atau responden memang **tidak ditanyai** karena tidak memenuhi syarat?

In [ ]:

# ===================== USIA PER LABEL, PER TARGET =====================
rows = []
for col in TARGETS:
    g = df.groupby(col)["Usia"].agg(["count", "min", "median", "max"])
    g.index = pd.MultiIndex.from_product([[col], g.index])
    rows.append(g)

display(pd.concat(rows))

In [ ]:

# ===================== PROPORSI 'TIDAK DAPAT PERTANYAAN' PER KELOMPOK USIA =====================
AGE_BINS = [0, 5, 15, 25, 35, 45, 55, 65, 120]
AGE_LABELS = ["0-4", "5-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65+"]
age_band = pd.cut(df["Usia"], bins=AGE_BINS, labels=AGE_LABELS, right=False)

share = pd.DataFrame({
    col: df.assign(g=age_band).groupby("g", observed=True)[col]
           .apply(lambda s: (s == "Tidak Dapat Pertanyaan").mean() * 100).round(1)
    for col in TARGETS
})
display(share)

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
share.plot(kind="bar", ax=ax)
ax.set_title("Proporsi 'Tidak Dapat Pertanyaan' per Kelompok Usia")
ax.set_xlabel("Kelompok Usia")
ax.set_ylabel("% dari responden pada kelompok itu")
ax.legend(title="Target", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:

# ===================== BATAS USIA EFEKTIF PER TARGET =====================
summary = []
for col in TARGETS:
    answered = df[df[col].isin(["Berisiko", "Tidak Berisiko", "Tidah Berisiko"])]
    unasked = df[df[col] == "Tidak Dapat Pertanyaan"]
    summary.append({
        "Target": col,
        "Usia min (dijawab)": answered["Usia"].min(),
        "Usia maks (tidak ditanya)": unasked["Usia"].max() if len(unasked) else "-",
        "N dijawab": f"{len(answered):,}",
        "N tidak ditanya": f"{len(unasked):,}",
    })

display(pd.DataFrame(summary))


### Temuan 1 — `Tidak Dapat Pertanyaan` adalah gerbang eligibilitas, bukan data hilang

Batas usianya bersih dan tidak ambigu:

- **Diabetes Melitus & Hipertensi** — hanya ditanyakan pada usia **>= 15 tahun**. Usia
  minimum pada kedua kelas berlabel tepat 15.
- **Kanker Paru** — hanya ditanyakan pada usia **>= 45 tahun**. Usia minimum pada kelas
  berlabel tepat 45, dan usia maksimum yang berlabel `Tidak Dapat Pertanyaan` tepat 44.
- **TBC** — ditanyakan pada **semua usia** (nol kejadian `Tidak Dapat Pertanyaan`).

**Konsekuensinya untuk seluruh proyek:**

1. Membuang baris `Tidak Dapat Pertanyaan` adalah **benar** — mereka bukan data hilang,
   melainkan responden yang memang di luar sasaran instrumen skrining.
2. Tetapi setiap dataset per-penyakit menjadi **subpopulasi dengan rentang usia berbeda**.
   Model Kanker Paru adalah model untuk usia 45+, bukan model untuk populasi umum.
3. Variabel `Usia` punya rentang terpotong di dalam tiap dataset, sehingga koefisien atau
   importance-nya tidak bisa dibandingkan langsung antar penyakit.
4. Analisis komorbiditas (berapa orang berisiko lebih dari satu penyakit) hanya bermakna
   pada usia **45+**, karena hanya di sana keempat pertanyaan sama-sama ditanyakan.


## 5. Struktur Kolom `id`

In [ ]:

# ===================== ANALISIS STRUKTUR ID =====================
n_rows, n_ids = len(df), df["id"].nunique()
per_id = df["id"].value_counts()

print(f"Jumlah baris      : {n_rows:,}")
print(f"Jumlah id unik    : {n_ids:,}")
print(f"Rata-rata baris per id : {per_id.mean():.1f}   (min {per_id.min()}, maks {per_id.max()})")
print()

var = df.groupby("id").agg(
    n=("Usia", "size"),
    n_jenis_kelamin=("Jeniskelamin", "nunique"),
    n_usia=("Usia", "nunique"),
)
print(f"id yang memuat KEDUA jenis kelamin : {int((var['n_jenis_kelamin'] > 1).sum()):,} dari {len(var):,}")
print(f"Rata-rata jumlah usia berbeda per id : {var['n_usia'].mean():.1f}")
print()

contoh = df[df["id"] == df["id"].iloc[0]]
print(f"Contoh — semua baris untuk id={df['id'].iloc[0]} (10 pertama):")
display(contoh[["id", "Jeniskelamin", "Usia", "segmen", "Tinggibadan", "Berat Badan"]].head(10))


### Temuan 2 — `id` adalah kunci sintetis hasil pengacakan, dan itu memang disengaja

Satu `id` yang sama memuat laki-laki maupun perempuan, dengan puluhan usia berbeda. Jadi
`id` **bukan** penanda individu.

Ini **bukan cacat data**. Kolom `id` sengaja diacak untuk melindungi data asli BPJS, dan
pola di atas justru menunjukkan anonimisasinya bekerja sebagaimana mestinya.

**Dua konsekuensi analitis:**

1. `id` tidak membawa informasi apa pun, tidak bisa dipakai sebagai kunci individu, dan
   harus dibuang sebelum modeling. Secara khusus, `GroupShuffleSplit` berbasis `id` tidak
   ada gunanya karena tidak ada struktur kunjungan-berulang yang bisa dikelompokkan.
2. **Deduplikasi tingkat individu menjadi mustahil.** Setiap baris harus diperlakukan
   sebagai satu catatan skrining yang berdiri sendiri, dan semua angka prevalensi nanti
   dilaporkan **per catatan skrining**, bukan per orang.


## 6. Pola Tinggi & Berat Badan Tertukar

In [ ]:

# ===================== DETEKSI POLA TERTUKAR =====================
h = df["Tinggibadan"].abs()
w = df["Berat Badan"].abs()

bmi_asis = w / (h / 100) ** 2          # IMT sebagaimana tercatat
bmi_swap = h / (w / 100) ** 2          # IMT jika tinggi & berat ditukar

implausible = ~bmi_asis.between(bs.BMI_MIN, bs.BMI_MAX)
recoverable = (
    implausible
    & bmi_swap.between(bs.BMI_MIN, bs.BMI_MAX)
    & w.between(*bs.GLOBAL_BOUNDS["tinggi"])   # berat tercatat masuk akal sbg TINGGI
    & h.between(*bs.GLOBAL_BOUNDS["berat"])    # tinggi tercatat masuk akal sbg BERAT
)

print(f"Baris dengan IMT di luar rentang wajar {bs.BMI_MIN}-{bs.BMI_MAX} : {int(implausible.sum()):,}")
print(f"  di antaranya menjadi wajar bila tinggi/berat ditukar     : {int(recoverable.sum()):,} "
      f"({recoverable.sum() / max(implausible.sum(), 1) * 100:.1f}%)")
print(f"  sisanya benar-benar tidak dapat diselamatkan             : {int((implausible & ~recoverable).sum()):,}")

# Simulasi urutan pembersihan (dipakai lagi di Bagian 7 & 10) supaya angka yang
# dilaporkan tidak tumpang tindih antar tahap.
h_fix = h.where(~recoverable, w)
w_fix = w.where(~recoverable, h)
bmi_fix = w_fix / (h_fix / 100) ** 2
survives_global = (
    h_fix.between(*bs.GLOBAL_BOUNDS["tinggi"])
    & w_fix.between(*bs.GLOBAL_BOUNDS["berat"])
    & bmi_fix.between(bs.BMI_MIN, bs.BMI_MAX)
)

print("\nContoh baris yang terindikasi tertukar:")
display(
    df.loc[recoverable, ["id", "Usia", "Tinggibadan", "Berat Badan", "Index Masa Tubuh"]]
      .assign(IMT_jika_ditukar=bmi_swap[recoverable].round(1))
      .head(12)
)


### Temuan 3 — mayoritas baris "rusak" sebenarnya hanya tertukar kolomnya

Pola `tinggi 54 cm / berat 159 kg` jelas merupakan tinggi dan berat yang tertukar posisi
saat input. Baris seperti ini **dapat diperbaiki**, bukan dibuang.

Yang perlu diperhatikan: deteksi ini **tidak bisa** mengandalkan pengecekan "apakah tinggi
dan berat masing-masing di luar rentang wajar", karena 54 masih merupakan tinggi yang sah
(untuk bayi) dan 159 masih merupakan berat yang sah (untuk orang obesitas). Keduanya lolos
bila diperiksa terpisah. Yang membongkar polanya adalah IMT hasil hitungannya, dan itulah
pendekatan yang dipakai di notebook Persiapan Data.


## 7. Batas Plausibilitas per Kelompok Usia

Batas global (tinggi 40-250 cm, berat 2-300 kg) berlaku untuk seluruh populasi dari bayi
sampai lansia, sehingga terpaksa sangat longgar. Akibatnya, nilai yang mustahil untuk usia
tertentu tetap lolos.

In [ ]:

# ===================== TINGGI BADAN vs USIA =====================
plaus = h.between(*bs.GLOBAL_BOUNDS["tinggi"]) & w.between(*bs.GLOBAL_BOUNDS["berat"])
sample = df[plaus].sample(min(40_000, int(plaus.sum())), random_state=42)

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
ax.scatter(sample["Usia"], sample["Tinggibadan"], s=2, alpha=0.15, color="steelblue")

for (lo, hi), b in bs.CHILD_BOUNDS.items():
    ax.plot([lo, hi + 1], [b["tinggi"][0]] * 2, color="crimson", lw=1.5)
    ax.plot([lo, hi + 1], [b["tinggi"][1]] * 2, color="crimson", lw=1.5)

ax.set_title("Tinggi Badan vs Usia (lolos batas global) — garis merah = batas wajar per usia")
ax.set_xlabel("Usia (tahun)")
ax.set_ylabel("Tinggi Badan (cm)")
ax.set_xlim(-1, 90)
plt.tight_layout()
plt.show()

In [ ]:

# ===================== PELANGGARAN BATAS PER USIA =====================
# Dihitung pada populasi yang SUDAH lolos tahap sebelumnya (perbaikan tukar +
# batas global + rentang IMT), sehingga angkanya mencerminkan tambahan baris yang
# hanya bisa ditangkap oleh batas per usia -- tidak tumpang tindih dengan tahap lain.
d_plaus = df[survives_global].assign(
    Tinggibadan=h_fix[survives_global], **{"Berat Badan": w_fix[survives_global]}
)
valid_age = bs.apply_child_bounds(d_plaus)

rows = []
for (lo, hi), b in bs.CHILD_BOUNDS.items():
    in_band = d_plaus["Usia"].between(lo, hi)
    bad = in_band & ~valid_age
    rows.append({
        "Kelompok usia": f"{lo}-{hi}",
        "Batas tinggi (cm)": f"{b['tinggi'][0]:.0f}-{b['tinggi'][1]:.0f}",
        "Batas berat (kg)": f"{b['berat'][0]:.0f}-{b['berat'][1]:.0f}",
        "N": f"{int(in_band.sum()):,}",
        "Tidak masuk akal": int(bad.sum()),
        "%": round(bad.sum() / max(in_band.sum(), 1) * 100, 2),
    })

display(pd.DataFrame(rows))
print(f"Total baris anak yang lolos batas global tapi mustahil untuk usianya: "
      f"{int((~valid_age).sum()):,}")

print("\nContoh terparah:")
display(
    d_plaus[~valid_age].nlargest(8, "Tinggibadan")[
        ["id", "Usia", "Tinggibadan", "Berat Badan", "Index Masa Tubuh"]
    ]
)


### Temuan 4 — batas global terlalu longgar untuk anak

Ada anak usia 0 tahun yang tercatat setinggi 197 cm. Nilai itu lolos batas global karena
197 cm adalah tinggi yang sah — untuk orang dewasa. Baris seperti ini merusak dua hal
sekaligus: statistik deskriptif pada kelompok anak, dan dataset TBC (satu-satunya target
yang mencakup semua usia).

Karena itu notebook Persiapan Data menerapkan batas tambahan **per kelompok usia** setelah
batas global, lalu membuang baris yang tetap mustahil.


## 8. Ketergantungan Label Target pada IMT

Bagian ini memeriksa hubungan antara label risiko dengan IMT. Idealnya hubungannya kuat
tetapi tidak deterministik — kalau deterministik, artinya label tersebut hanyalah hasil
perhitungan ulang dari IMT, bukan informasi klinis baru.

In [ ]:

# ===================== P(Berisiko | IMT) PER TARGET =====================
# Dihitung pada baris dengan antropometri wajar dan usia dewasa (> 20), supaya
# aturan persentil anak tidak mencampuri hasilnya.
bmi_ok = bmi_asis.between(bs.BMI_MIN, bs.BMI_MAX) & plaus
adult = df["Usia"] > bs.CHILD_MAX_AGE

rows = []
for col in TARGETS:
    lab = df[col].replace({"Tidah Berisiko": "Tidak Berisiko"})
    m = bmi_ok & adult & lab.isin(["Berisiko", "Tidak Berisiko"])
    sub_hi = m & (bmi_asis > 25)
    sub_lo = m & (bmi_asis <= 25)
    rows.append({
        "Target": col,
        "N dewasa": f"{int(m.sum()):,}",
        "P(Berisiko | IMT > 25)": round((lab[sub_hi] == "Berisiko").mean(), 4),
        "P(Berisiko | IMT <= 25)": round((lab[sub_lo] == "Berisiko").mean(), 4),
        "IMT maks pd 'Tidak Berisiko'": round(bmi_asis[m & (lab == "Tidak Berisiko")].max(), 1),
        "Pengecualian di IMT>25": int((lab[sub_hi] != "Berisiko").sum()),
    })

display(pd.DataFrame(rows))

In [ ]:

# ===================== SISA SINYAL PADA IMT <= 25 =====================
# Kalau label memang cuma aturan IMT, tidak akan ada pola tersisa di bawah ambang.
band = pd.cut(df["Usia"], [20, 29, 39, 49, 59, 69, 200],
              labels=["21-29", "30-39", "40-49", "50-59", "60-69", "70+"])

resid = {}
for col in TARGETS:
    lab = df[col].replace({"Tidah Berisiko": "Tidak Berisiko"})
    m = bmi_ok & adult & lab.isin(["Berisiko", "Tidak Berisiko"]) & (bmi_asis <= 25)
    resid[col] = (
        df[m].assign(g=band[m], lab=lab[m])
             .groupby("g", observed=True)["lab"]
             .apply(lambda s: (s == "Berisiko").mean() * 100).round(1)
    )

display(pd.DataFrame(resid).rename_axis("Kelompok usia"))
print("(% Berisiko di antara responden dewasa ber-IMT <= 25)")


### Temuan 5 — label Diabetes praktis merupakan aturan `IMT > 25`

Pada responden dewasa, hampir **setiap** orang ber-IMT di atas 25 diberi label `Berisiko
Diabetes Melitus`, dengan jumlah pengecualian yang sangat kecil. Hipertensi menunjukkan
hubungan yang kuat tetapi **tidak** deterministik, sedangkan Kanker Paru dan TBC hampir
tidak terpengaruh IMT sama sekali.

**Implikasinya besar dan harus dibawa ke seluruh laporan:**

1. Model diabetes yang dilatih memakai IMT sebagai fitur pada dasarnya sedang
   **merekonstruksi aturan penilaian kuesioner**, bukan memprediksi risiko penyakit.
   Akurasi tinggi pada target ini tidak boleh dibaca sebagai keberhasilan prediksi.
2. Prevalensi "berisiko diabetes" yang dilaporkan pada dasarnya adalah pernyataan ulang
   dari prevalensi overweight, sehingga tidak boleh disajikan sebagai temuan epidemiologis
   yang berdiri sendiri.
3. Sisa sinyal yang benar-benar informatif ada pada kelompok ber-IMT <= 25, di mana
   proporsi berisiko naik konsisten seiring usia. Bagian inilah yang layak dimodelkan.


## 9. Korelasi Antar Variabel

In [ ]:

# ===================== KORELASI =====================
fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

# Kiri: antar variabel numerik, dihitung hanya pada baris yang wajar supaya
# outlier ekstrem tidak menghancurkan koefisiennya.
clean_num = df.loc[bmi_ok, NUM_COLS]
sns.heatmap(clean_num.corr(), annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, square=True, ax=axes[0])
axes[0].set_title("Korelasi Variabel Numerik\n(baris dengan antropometri wajar)")

# Kanan: antar target, dihitung pada usia 45+ di mana keempat pertanyaan ditanyakan.
mask45 = df["Usia"] >= 45
risk = pd.DataFrame({
    c: df.loc[mask45, c].replace({"Tidah Berisiko": "Tidak Berisiko"})
         .map({"Tidak Berisiko": 0, "Berisiko": 1})
    for c in TARGETS
}).dropna()
sns.heatmap(risk.corr(), annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, square=True, ax=axes[1])
axes[1].set_title(f"Korelasi Antar Risiko Penyakit\n(usia 45+, n={len(risk):,})")

plt.tight_layout()
plt.show()


## 10. Ringkasan Masalah & Pemetaan ke Tahap Pembersihan

In [ ]:

# ===================== RINGKASAN TEMUAN =====================
# Angka baris dihitung SECARA BERURUTAN mengikuti tahapan pembersihan, jadi
# tiap baris tabel di bawah saling lepas (tidak ada baris yang dihitung dua kali).
n_repaired = int(recoverable.sum())
n_drop_global = int((~survives_global).sum())
n_drop_age = int((~valid_age).sum())
n_sisa = int(survives_global.sum()) - n_drop_age

ringkasan = pd.DataFrame([
    ["Typo label target", f"{int((df[TARGETS] == 'Tidah Berisiko').sum().sum()):,} baris",
     "Diperbaiki", "Standardisasi label di SEMUA kolom target"],
    ["Tinggi/berat tertukar", f"{n_repaired:,} baris",
     "Diperbaiki", "Tukar kembali kedua kolom"],
    ["Antropometri / IMT tetap mustahil", f"{n_drop_global:,} baris",
     "Dibuang", "Batas global + rentang IMT 8-70"],
    ["Mustahil untuk usianya (anak)", f"{n_drop_age:,} baris",
     "Dibuang", "Batas tambahan per kelompok usia"],
    ["IMT asli tercemar error input", "seluruh kolom",
     "Dihitung ulang", "IMT dari tinggi & berat yang sudah bersih"],
    ["'Tidak Dapat Pertanyaan'", f"{int((df[TARGETS] == 'Tidak Dapat Pertanyaan').sum().sum()):,} sel",
     "Dibuang per target", "Tidak eligible, bukan data hilang"],
    ["id teracak", f"{n_ids:,} id untuk {n_rows:,} baris",
     "Dibuang saat modeling", "Deduplikasi individu tidak mungkin"],
], columns=["Masalah", "Skala", "Perlakuan", "Tindakan di notebook Persiapan Data"])

display(ringkasan)

print(f"Perkiraan baris akhir setelah pembersihan: {n_rows:,} - {n_drop_global:,} - "
      f"{n_drop_age:,} = {n_sisa:,}")


### Kesimpulan

Data ini **tidak** punya masalah kelengkapan — tidak ada satu pun nilai kosong. Masalahnya
seluruhnya berupa **nilai yang salah namun tersimpan sebagai angka yang valid**, sehingga
hanya terdeteksi lewat pemeriksaan plausibilitas, bukan lewat pemeriksaan missing value.

Tiga temuan yang paling menentukan arah proyek:

1. **Gerbang eligibilitas** (Bagian 4) — setiap target mencakup rentang usia yang berbeda.
   Setiap model dan setiap angka prevalensi harus menyebutkan rentang usia berlakunya.
2. **`id` teracak** (Bagian 5) — analisis tingkat individu tidak mungkin; semua angka
   dilaporkan per catatan skrining.
3. **Label diabetes bersifat aturan** (Bagian 8) — performa model pada target ini tidak
   boleh dibaca sebagai kemampuan prediksi.

Selanjutnya: `Persiapan_Data_Revised.ipynb`.